In [1]:
import pandas as pd
from tqdm import tqdm 
import scanpy as sc
import sys
import os 
import logging 
from omegaconf import OmegaConf, DictConfig
from hydra import initialize, compose
from scipy.spatial.distance import cdist
from sklearn.preprocessing import LabelEncoder
import torch
import numpy as np
from pathlib import Path 
import torch.nn.functional as F

from sc_exp_design.constants import DataFields
sys.path.insert(0, "../../../shared_utils")
from experiment_utils import (
    get_forward_model,
    get_target_dict, 
    get_loss_fn
)

logger = logging.getLogger(__name__)

In [56]:
exp_245_loop2 = np.array([9.2, 2.2, 997.3, 1120.2, 0.0, 57.2, 0.0, 0.0, 0.1, 0.6, 20.0, 14.0])
exp_245_loop3_ly15 = np.array([9.2, 2.2, 997.3, 1120.2, 0.0, 57.2, 0.0, 0.0, 0.1, 0.6, 20.0, 14.0, 0.0, 0.0, 15.0])
exp_245_loop3_ly1p5 = np.array([9.2, 2.2, 997.3, 1120.2, 0.0, 57.2, 0.0, 0.0, 0.1, 0.6, 20.0, 14.0, 0.0, 0.0, 1.5])

exp_245_loop2 = np.log1p(exp_245_loop2)
exp_245_loop3_ly15 = np.log1p(exp_245_loop3_ly15)
exp_245_loop3_ly1p5 = np.log1p(exp_245_loop3_ly1p5)

exp_245_loop2 = exp_245_loop2[None, :]
exp_245_loop3_ly15 = exp_245_loop3_ly15[None, :] 
exp_245_loop3_ly1p5 = exp_245_loop3_ly1p5[None, :]

In [53]:
def uncertainty_quantification(
    forward_model,
    base_cfg,
    annotation, 
    paths, 
    constraints,
    X_candidates, 
    target_cell_types, 
    n_noise_samples,
    n_populations, 
    cell_type_column):
    
    protocol_columns = base_cfg["annotation"]["protocol_columns"]

    ct_le = LabelEncoder()
    ct_values = target_prediction_model.train_data.adata.obs[cell_type_column].values
    ct_le.fit(ct_values)
    classes = ct_le.classes_.tolist()  # List of cell types in alphabetical order 
    print(classes)

    # Take cell type index 
    cell_type_index = classes.index(target_cell_types)
    ct_safe_string = target_cell_types.replace("/", ":") # cell type dir

    X_candidates = torch.from_numpy(X_candidates).float().to("cuda")
    X_candidates_add = X_candidates.unsqueeze(1).repeat(1, n_noise_samples, 1)  # no_candidates x n_noise_sample x candidate_dim
    condition_repr = next(iter(forward_model.forward_model.train_data.data.perturbation_covariates))
    batch_dict = {DataFields.PERTURBATION_DATA: {condition_repr: X_candidates_add}}

    forward_out = forward_model.predict(
        batch_dict,
        no_grad=True,
        fix_noise=False,
        num_time_steps=100
    )
    
    y_pred = forward_out["target_prediction_data"][cell_type_column].view(X_candidates_add.shape[0],
                                                                     n_populations,
                                                                     -1,
                                                                     len(classes))  
    y_pred_softmax = F.softmax(y_pred, dim=-1)

    # Take the mean proportions 
    y_pred_mean = y_pred.mean(2)  # no_candidates x no_populations x no_cell_types 
    y_pred_softmax = y_pred_softmax.mean(2)  # no_candidates x no_populations x no_cell_types 
    print(y_pred_softmax[:, :, 21].mean())

    # Collect cell type of interest 
    y_pred_ct_std = y_pred_softmax[..., cell_type_index].std(1).detach().cpu().numpy()  # standard deviation prob cell type of interest 
    # Calculate loss
    y_target_ct = np.zeros((1, len(classes)))
    y_target_ct[:, cell_type_index] = 1.0
    y_target_ct = torch.from_numpy(y_target_ct)
    y_target_ct = y_target_ct.unsqueeze(0).to("cuda")  # 1 x 1 x no_cell_types
    
    logger.info("Compute loss function")
    loss_fn = lambda pred, target: -torch.sum(target*torch.nn.functional.log_softmax(pred, dim=-1), dim=-1)
    with torch.no_grad():
        loss = loss_fn(y_pred_mean, y_target_ct)  # no_candidates x no_populations

    loss_std = loss.std(1).detach().cpu().numpy()  # no_candidates
    return loss_std

Loop 3

In [22]:
base_config_path = "../../../inverse/loss_guidance/config/"
base_config_name = "run_inverse"
annotation = "bloodplus_loop3"
paths = "loop3_replicate"
constraints = "default_all_cols_loop3"
target_cell_types = "preProB"
n_noise_samples = 5000
n_populations = 10
cell_type_column = "cell_type" 

# Initialize paths 
with initialize(config_path=base_config_path, version_base=None):
    base_cfg = compose(
        config_name=base_config_name,
        overrides=[f"paths={paths}", 
                   f"annotation={annotation}", 
                   f"constraints={constraints}"])
base_cfg.loss.cell_type_column = cell_type_column

# Collect the forward model
# forward_model, (_, target_prediction_model) = get_forward_model(base_cfg, logger=logger)

In [26]:
uncertainty_quantification(
    forward_model, 
    base_cfg,
    annotation, 
    paths, 
    constraints,
    exp_245_loop3_ly1p5, 
    target_cell_types, 
    n_noise_samples,
    n_populations, 
    cell_type_column)

['CD14Mono', 'CD16Mono', 'CyclingPro1', 'CyclingPro2', 'Early_GMP', 'EosBasMast1', 'EosBasMast2', 'EosBasMast3', 'EryPro', 'GMP-Neutro', 'GMP_cycling', 'HSCs', 'LMPP-CLP', 'MLP', 'MPP', 'MPP_MgkEry', 'MgKEryPro1', 'MgKEryPro2', 'ProB', 'earlyMPP', 'late_MgkPro', 'preProB', 'pre_cDC1', 'pre_cDC2']
tensor(0.0017, device='cuda:0', grad_fn=<MeanBackward0>)


Loop 1

In [27]:
base_config_path = "../../../inverse/loss_guidance/config/"
base_config_name = "run_inverse"
annotation = "bloodplus"
paths = "bloodplus_fm"
constraints = "default_all_cols"
target_cell_types = "preProB"
n_noise_samples = 5000
n_populations = 10
cell_type_column = "cell_type" 

# Initialize paths 
with initialize(config_path=base_config_path, version_base=None):
    base_cfg = compose(
        config_name=base_config_name,
        overrides=[f"paths={paths}", 
                   f"annotation={annotation}", 
                   f"constraints={constraints}"])
base_cfg.loss.cell_type_column = cell_type_column

forward_model, (_, target_prediction_model) = get_forward_model(base_cfg, logger=logger)

In [55]:
uncertainty_quantification(
    forward_model, 
    base_cfg,
    annotation, 
    paths, 
    constraints,
    exp_245_loop2, 
    target_cell_types, 
    n_noise_samples,
    n_populations, 
    cell_type_column)

['CD14Mono', 'CD16Mono', 'CyclingPro1', 'CyclingPro2', 'Early_GMP', 'EosBasMast1', 'EosBasMast2', 'EosBasMast3', 'EryPro', 'GMP-Neutro', 'GMP_cycling', 'HSCs', 'LMPP-CLP', 'MLP', 'MPP', 'MPP_MgkEry', 'MgKEryPro1', 'MgKEryPro2', 'ProB', 'earlyMPP', 'late_MgkPro', 'preProB', 'pre_cDC1', 'pre_cDC2']
tensor(0.0062, device='cuda:0', grad_fn=<MeanBackward0>)
tensor([4.0978], device='cuda:0', dtype=torch.float64)
